In [20]:
import ee 
import geemap 

# Initialize Earth Engine
ee.Initialize(project = 'wind-field-estimation-497821') 

# Define area (Tamil Nadu coast box)
aoi = ee.Geometry.Rectangle([68.0,21.0,70.0,23.0]) 

# Load Sentinel-1 SAR collection - EXPANDED TO A 1-MONTH WINDOW
collection = ( 
    ee.ImageCollection('COPERNICUS/S1_GRD') 
    .filterBounds(aoi) 
    .filterDate('2024-01-01', '2024-01-31')  # Changed from Jan 10 to Jan 31
    .filter(ee.Filter.eq('instrumentMode', 'IW')) 
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')) 
    .select('VV') 
) 

# SAFE CHECK: See how many images match your search
count = collection.size().getInfo()
print("Images found in collection:", count)

# Only process the image if the collection is NOT empty
if count > 0:
    # Take first image 
    image = collection.first() 
    print("Image ID loaded:", image.id().getInfo()) 

    # Create and display map 
    Map = geemap.Map() 
    Map.addLayer(image, {'min': -25, 'max': 0}, 'Sentinel-1 VV') 
    Map.centerObject(aoi, 8) 
    Map
else:
    print("❌ Still no images found. Try expanding the date range further or adjusting the AOI coordinates.")


Images found in collection: 13
Image ID loaded: S1A_IW_GRDH_1SDV_20240101T133518_20240101T133547_051914_0645AF_3E82


In [16]:
# Create the map canvas
Map = geemap.Map() 

# Add your VV polarization radar data layer
Map.addLayer(image, {'min': -25, 'max': 0}, 'Sentinel-1 VV') 

# Zoom directly to your Tamil Nadu coast box
Map.centerObject(aoi, 8) 

# Force Jupyter to display the map
Map


Map(center=[22.000678814980056, 69.00000000000001], controls=(WidgetControl(options=['position', 'transparent_…

In [17]:
print(collection.size().getInfo())

13


In [18]:
print(image.bandNames().getInfo())

['VV']


That black-white image is NOT:
photo
RGB image
visible light

It is:
microwave radar reflection intensity
called:
backscatter

VV is radar polarisation
vertical transmit vertical recieve, used for our thing - which is ocean wind estimation
VH is horizontal, used for storms ships etc. irrelevant rn

In [19]:
print("Number of images:", collection.size().getInfo())

print("\nBand names:")
print(image.bandNames().getInfo())

print("\nImage properties:")
props = image.toDictionary().getInfo()

for key in list(props.keys())[:20]:
    print(key)

Number of images: 13

Band names:
['VV']

Image properties:
GRD_Post_Processing_facility_country
GRD_Post_Processing_facility_name
GRD_Post_Processing_facility_org
GRD_Post_Processing_facility_site
GRD_Post_Processing_software_name
GRD_Post_Processing_software_version
GRD_Post_Processing_start
GRD_Post_Processing_stop
S1TBX_Calibration_vers
S1TBX_IO_Orbit_Ephemeris_vers
S1TBX_SAR_Processing_vers
SLC_Processing_facility_country
SLC_Processing_facility_name
SLC_Processing_facility_org
SLC_Processing_facility_site
SLC_Processing_software_name
SLC_Processing_software_version
SLC_Processing_start
SLC_Processing_stop
SNAP_Graph_Processing_Framework_GPF_vers


In [7]:
print(image.projection().getInfo())

{'type': 'Projection', 'crs': 'EPSG:32644', 'transform': [10, 0, 215552.82841487997, 0, -10, 1635689.3937842965]}
